In [1]:
# This part is for 

from pathlib import Path
path_spec_data=Path.cwd().parent.parent/"spec_data"
path_benchmark_data=Path.cwd().parent.parent/"benchmark_group_size_not_convert"

path_spec_data.mkdir(parents=True, exist_ok=True)
path_benchmark_data.mkdir(parents=True, exist_ok=True)

In [ ]:
cache_list_threshold=1_000_000

ion_mode=[1, -1]

steps=['build', 'open_search', 'neutral_loss_search', 'hybrid_search']

num_per_group=[ 
    1_000_000, 3_000_000,  
    10_000_000, 30_000_000, 
    100_000_000, 300_000_000
    ]

dynamic_script_path="15-2_dynamic_entropy_search_group_size_hybrid_fast_update_mode_every_step_not_convert.py"

In [3]:
import msgpack
import subprocess
import os
import shutil
import pickle
from typing import Union

def run_usrbintime_by_arguments(
          arguments:list[str], 
          if_output:bool=False, 
          output_memory_file:Union[str,Path]=None, 
          output_time_file:Union[str, Path]=None):
    
    # arguments: script_path, str(charge), step
    command=["/usr/bin/time","-v","python"] + arguments

    if if_output: # Output to files as record
        with open(output_memory_file, "w") as f1, open(output_time_file, "w") as f2:
            subprocess.run(command, stderr=f1, stdout=f2, cwd=Path.cwd(), env=os.environ.copy())

    else: # Output is not needed
         
        subprocess.run(command, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, cwd=Path.cwd(), env=os.environ.copy())
        
    return

def perform_search_and_record_one_by_one(
          script_path:Path, 
          search_type:str,
          charge:int,
          library_size:int,
          step:str,
          query_spectra_list:list,
          num_per_group:int,
          cache_list_threshold:int,
          method:str,
          if_output:bool=True, 
          ):
    
    for i, spec in enumerate(query_spectra_list):
        # Generate temp query file to perform search one by one
        temp_query_spec=path_spec_data/f"benchmark_spec/query_spectra-charge_{charge}_temp.pkl"
        with open(temp_query_spec, "wb") as temp1:

            # Write temp query pickle file
            pickle.dump(spec, temp1)

        if search_type is not None:
            arguments=[script_path, str(charge), str(num_per_group), str(cache_list_threshold), temp_query_spec, step]
            output_memory_file=path_benchmark_data/f"{method}_{search_type}_{charge}_{library_size}_group_size_{num_per_group}_memory_usage_{step}_step_query_{i}.txt"
            output_time_file=path_benchmark_data/f"{method}_{search_type}_{charge}_{library_size}_group_size_{num_per_group}_compare_time_{step}_step_query_{i}.txt"
        elif search_type is None:
            arguments=[script_path, str(charge), temp_query_spec, step]
            output_memory_file=path_benchmark_data/f"{method}_{charge}_{library_size}_memory_usage_{step}_step_query_{i}.txt"
            output_time_file=path_benchmark_data/f"{method}_{charge}_{library_size}_compare_time_{step}_step_query_{i}.txt"
        # Perform search and record memory usage and time
        run_usrbintime_by_arguments(arguments=arguments, if_output=if_output, output_memory_file=output_memory_file, output_time_file=output_time_file)


for charge in ion_mode:
    for group_size in num_per_group:
        for step in steps:
            if step=='build':
                # remove the old index
                path_comparison_dynamic_data=Path.cwd().parent.parent/f"comparison_data/dynamic/charge-{charge}"
                if path_comparison_dynamic_data.exists():
                    shutil.rmtree(path_comparison_dynamic_data)

                arguments=[dynamic_script_path, str(charge), str(group_size), str(cache_list_threshold), step]
                output_memory_file=path_benchmark_data/f"dynamic_fast_update_{charge}_memory_usage_options_{group_size}_num_per_group_not_convert.txt"
                output_time_file=path_benchmark_data/f"dynamic_fast_update_{charge}_compare_time_options_{group_size}_num_per_group_not_convert.txt"
                run_usrbintime_by_arguments(arguments=arguments,
                                            if_output=True,
                                            output_memory_file=output_memory_file,
                                            output_time_file=output_time_file)

            elif step[-6:]=='search':
                query_pkl=path_spec_data/f"benchmark_spec/query_spectra-charge_{charge}-number_100.pkl"
                query_spectra=pickle.loads(open(query_pkl, "rb").read())

                perform_search_and_record_one_by_one(
                    script_path=dynamic_script_path,
                    search_type='fast_update',
                    charge=charge,
                    library_size=100_000_000,
                    step=step,
                    query_spectra_list=query_spectra,
                    num_per_group=group_size,
                    cache_list_threshold=cache_list_threshold,
                    method='dynamic',
                    if_output=True
                )
